<div align="center">

# 🚕 Yandex Go Analytics

## 4 · Data Cleaning

*Turn raw Bronze records into trustworthy Silver tables.*

![layer](https://img.shields.io/badge/layer-silver-9E9E9E)
![engine](https://img.shields.io/badge/engine-Structured%20Streaming-E25A1C)
![quality](https://img.shields.io/badge/quality-validated-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) |

**Progress** &nbsp; ▰▰▰▰▱▱▱▱  **4 / 8**

---

## 🎯 Task requirements

Read from Bronze, apply the transformations below and save the results to the `silver` schema.

### 🚕 `silver.taxi`
- [x] Keep rows with `passenger_count > 0`, `trip_distance > 0`, `fare_amount > 0`
- [x] Map `payment_type` to a new description column: `1 = Credit Card`, `2 = Cash`, otherwise `Other`
- [x] Add an ingestion timestamp (`current_timestamp()`) and drop duplicates

### 🧑‍✈️ `silver.drivers`
- [x] Keep rows with `experience > 0` and `rating` between `1.0` and `5.0`
- [x] Add an ingestion timestamp and drop duplicates by the driver's ID

### 👤 `silver.users`
- [x] Filter out records with a null name
- [x] Replace nulls in the second business attribute with `"Unknown"`
- [x] Add an ingestion timestamp and drop duplicates

## 🧩 How it is solved

| Table | Filter | Transformation | De-duplication key |
|:--|:--|:--|:--|
| `silver.taxi` | `passenger_count > 0` ∧ `trip_distance > 0` ∧ `fare_amount > 0` | `payment_type_description` (`CASE WHEN`), `_processed_at` | `user_id`, `driver_id`, `pickup_datetime`, `dropoff_datetime` |
| `silver.drivers` | `experience > 0` ∧ `1.0 ≤ rating ≤ 5.0` | `_processed_at` | `id` |
| `silver.users` | `full_name IS NOT NULL` | `email` → `COALESCE(email, 'Unknown')`, `_processed_at` | `email` |

> 📝 The ingestion timestamp column is named `_processed_at`; the users' schema has `full_name` / `email`
> (see the *naming notes* in the project overview).

Every service reads its Bronze table as a **Delta stream** (`ignoreChanges`) and appends to Silver with
`availableNow` + a checkpoint — so only *new* Bronze rows are processed on every run.

```text
 bronze.taxi    ──▶ TripSilverTransformationService   ──▶ silver.taxi
 bronze.drivers ──▶ DriverSilverTransformationService ──▶ silver.drivers
 bronze.users   ──▶ UserSilverTransformationService   ──▶ silver.users
```

## 1️⃣ Connect to Spark

In [ ]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-data-cleaning")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

## 2️⃣ Anomalies in Bronze

Before cleaning, count how many records violate each rule. These are the rows Silver will remove.

In [ ]:
bronze_taxi = spark.table(f"{CATALOG}.bronze.taxi")
bronze_drivers = spark.table(f"{CATALOG}.bronze.drivers")
bronze_users = spark.table(f"{CATALOG}.bronze.users")


def violations(df, rules: dict):
    """One-row DataFrame with the number of rows breaking each rule."""
    return df.agg(*[F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in rules.items()])


print("bronze.taxi")
display(
    violations(
        bronze_taxi,
        {
            "zero_passengers": F.col("passenger_count") <= 0,
            "zero_distance": F.col("trip_distance") <= 0,
            "zero_fare": F.col("fare_amount") <= 0,
        },
    )
)

print("bronze.drivers")
display(
    violations(
        bronze_drivers,
        {
            "zero_experience": F.col("experience") <= 0,
            "rating_below_1": F.col("rating") < 1.0,
            "rating_above_5": F.col("rating") > 5.0,
        },
    )
)

print("bronze.users")
display(
    violations(
        bronze_users,
        {"null_name": F.col("full_name").isNull(), "null_email": F.col("email").isNull()},
    )
)

## 3️⃣ Run the Silver transformations

```text
 filter  ─▶  decode / enrich  ─▶  add _processed_at  ─▶  drop duplicates  ─▶  append to Silver
```

In [ ]:
from src.core.etl.silver import (
    DriverSilverTransformationService,
    TripSilverTransformationService,
    UserSilverTransformationService,
)

print("1. bronze.taxi    ->  silver.taxi")
TripSilverTransformationService().run(spark)

print("\n2. bronze.drivers ->  silver.drivers")
DriverSilverTransformationService().run(spark)

print("\n3. bronze.users   ->  silver.users")
UserSilverTransformationService().run(spark)

print("\n✅ Silver layer is up to date")

---

## ✅ Checkpoint

- [x] `silver.taxi` — invalid trips removed, payment type decoded, duplicates dropped
- [x] `silver.drivers` — valid experience and rating only, one row per driver
- [x] `silver.users` — named users only, missing e-mail replaced by `Unknown`, duplicates dropped

> **Next up ▶ [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb)** — aggregate Silver into the Gold metrics and optimise the tables.

<div align="center"><sub>Yandex Go Analytics · notebook 4 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>